# 10 · Streaming, Tracing and Hooks

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

Three ways to see inside a run: **streaming** shows it to the *user* as it
happens, **tracing** records it for *developers* afterwards, and **hooks** let
*your code* react to each step (logging, metrics, alerts). This notebook covers
all three, with tracing kept entirely **local**, so nothing leaves your machine.

**You will learn**
- `Runner.run_streamed` and the three event types: raw tokens, run items, agent changes
- Building a live UI stream: tokens for the answer, status lines for tools and handoffs
- Tracing: traces, spans, grouping runs with `trace()`, and what gets recorded
- A **local trace processor** that prints a span tree, instead of uploading to OpenAI
- **Hooks** (`RunHooks`, `AgentHooks`) for logging and metrics
- Sensitive data in traces

**Prerequisites:** notebooks 01–04 and 06.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import time
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, function_tool, trace, RunHooks, RunConfig
from common import get_model

model = get_model()


@function_tool
def get_weather(city: str) -> str:
    """Current weather for a city."""
    time.sleep(0.5)
    return {'pune': '31°C humid', 'oslo': '4°C rain'}.get(city.lower(), '20°C clear')

travel = Agent(name='Travel', model=model, tools=[get_weather],
               instructions='Answer travel questions. Use tools for weather. Be brief.')

## 1 · Why: three audiences, three tools

| Audience | Needs | Tool |
|---|---|---|
| The **user** | words as they're generated, and "checking the weather…" status | streaming |
| The **developer** | afterwards: what ran, in what order, how long, with which inputs | tracing |
| **Your code** | a callback at each step: count tokens, log tool calls, alert on errors | hooks |

## 2 · Streaming a run

`Runner.run_streamed(agent, input)` returns immediately with a
`RunResultStreaming`. Iterate `result.stream_events()` to receive events **as
they happen**; once the loop ends, `result.final_output` is available as usual.

There are three event types:

| `event.type` | Carries | Use |
|---|---|---|
| `raw_response_event` | raw model output chunks, e.g. `ResponseTextDeltaEvent` with `.delta` (a few characters) | typing effect |
| `run_item_stream_event` | a finished **item**: `tool_called`, `tool_output`, `message_output_created`, `handoff_requested`, … | status lines, logs |
| `agent_updated_stream_event` | the new active agent (at start and after handoffs) | "you're now talking to Billing" |

In [ ]:
result = Runner.run_streamed(travel, 'Should I pack an umbrella for Oslo?')
t0 = time.perf_counter()
first_token = None

async for event in result.stream_events():
    if event.type == 'agent_updated_stream_event':
        print(f'[agent: {event.new_agent.name}]')
    elif event.type == 'run_item_stream_event':
        if event.name == 'tool_called':
            print(f'[checking: {event.item.raw_item.name} {event.item.raw_item.arguments}]')
        elif event.name == 'tool_output':
            print(f'[got: {event.item.output}]')
    elif event.type == 'raw_response_event' and isinstance(event.data, ResponseTextDeltaEvent):
        first_token = first_token or time.perf_counter() - t0
        print(event.data.delta, end='', flush=True)

print(f'\n\nfirst token after {first_token:.1f}s · total {time.perf_counter() - t0:.1f}s')
print('final_output is available too:', result.final_output[:60], '…')

The user saw a status line while the tool ran, then the answer appeared word by
word. Perceived latency is time-to-first-token, not total time.

**Streaming and approvals.** If an approval-gated tool (notebook 08) is called
during a streamed run, the stream ends and `result.interruptions` is set, just
like with `Runner.run`. Resume with `Runner.run_streamed(agent, state)`.

## 3 · Tracing: what it records

Every run is recorded as a **trace** made of nested **spans**:

```
 trace "trip planning"                       ← one workflow (can contain several runs)
 └─ task / agent "Travel"                    ← an agent's turn in the run
     ├─ generation (model call)               ← model, input, output, token usage
     ├─ function  get_weather                 ← tool name, arguments, result
     ├─ generation (model call)
     └─ … handoff / guardrail spans when they happen
```

By default the SDK **uploads** traces to OpenAI's dashboard, which needs an
OpenAI key. `common.py` disabled that. Instead, we can install our own
**trace processor**: an object whose methods are called as traces and spans
start and end. It can print, write to a file, or forward to any observability
tool, and nothing leaves the machine.

In [ ]:
from agents import TracingProcessor, set_trace_processors, set_tracing_disabled


class LocalTraceRecorder(TracingProcessor):
    # Collects finished spans in memory and prints them as a tree.

    def __init__(self):
        self.spans = []

    def on_trace_start(self, trace): self.trace_name = trace.name
    def on_trace_end(self, trace): pass
    def on_span_start(self, span): pass
    def on_span_end(self, span): self.spans.append(span)
    def shutdown(self): pass
    def force_flush(self): pass

    def print_tree(self):
        children = {}
        for s in self.spans:
            children.setdefault(s.parent_id, []).append(s)
        def label(s):
            d = s.span_data
            name = getattr(d, 'name', None) or getattr(d, 'model', None) or ''
            ms = 0
            if s.started_at and s.ended_at:
                from datetime import datetime
                ms = (datetime.fromisoformat(s.ended_at) - datetime.fromisoformat(s.started_at)).total_seconds() * 1000
            return f'{d.type:<11} {name:<18} {ms:7.0f} ms'
        def walk(parent, depth):
            for s in sorted(children.get(parent, []), key=lambda s: s.started_at or ''):
                print('   ' * depth + '└─ ' + label(s))
                walk(s.span_id, depth + 1)
        print(f'trace: {self.trace_name}')
        walk(None, 0)


recorder = LocalTraceRecorder()
set_trace_processors([recorder])        # REPLACE the default OpenAI exporter with ours
set_tracing_disabled(False)             # turn tracing back on (locally now)

with trace('trip planning'):            # group several runs into ONE trace
    await Runner.run(travel, 'Weather in Pune?')
    await Runner.run(travel, 'And in Oslo?')

recorder.print_tree()

Both runs appear under one trace because they ran inside `with trace(...)`.
Without it, each run gets its own trace. For each span you can see what it was
and how long it took: two model calls around a tool call per run, and the
tool's 0.5 s delay visible in its span.

Spans carry much more than we printed: a generation span holds the model's
input, output and token usage, and a function span holds the tool's arguments and
result. That's exactly what you need to debug "why did it do that?", and exactly
why traces are sensitive:

**Sensitive data.** By default spans include model inputs/outputs and tool
arguments/results. `RunConfig(trace_include_sensitive_data=False)` keeps the
span structure but drops those payloads. Decide what may be recorded before
sending traces to any hosted service.

**Where traces can go.** `add_trace_processor(p)` adds a processor next to the
default exporter; `set_trace_processors([...])` replaces it. Integrations exist
for many observability tools, and with an OpenAI key the default exporter shows
traces in the OpenAI dashboard.

## 4 · Hooks: your code at each step

Streaming is for the UI and tracing for later analysis. **Hooks** run your code
*during* the run, at each lifecycle event. Subclass `RunHooks` (for a whole run)
or `AgentHooks` (attached to one agent) and override what you need:

| Hook | Called when |
|---|---|
| `on_agent_start` / `on_agent_end` | an agent starts / produces its final output |
| `on_llm_start` / `on_llm_end` | a model call begins / returns |
| `on_tool_start` / `on_tool_end` | a tool is about to run / has returned |
| `on_handoff` | control passes to another agent |

A small metrics collector, the kind of thing you'd send to a dashboard:

In [ ]:
class Metrics(RunHooks):
    def __init__(self):
        self.model_calls = 0
        self.tool_calls = []
        self.tokens = 0
        self._tool_started = {}          # tool name → queue of start times (calls can overlap)

    async def on_llm_end(self, context, agent, response):
        self.model_calls += 1
        self.tokens += response.usage.total_tokens

    async def on_tool_start(self, context, agent, tool):
        self._tool_started.setdefault(tool.name, []).append(time.perf_counter())

    async def on_tool_end(self, context, agent, tool, result):
        took = time.perf_counter() - self._tool_started[tool.name].pop(0)
        self.tool_calls.append((tool.name, f'{took:.2f}s'))


metrics = Metrics()
await Runner.run(travel, 'Compare the weather in Pune and Oslo.', hooks=metrics)
print('model calls:', metrics.model_calls, '| tokens:', metrics.tokens, '| tools:', metrics.tool_calls)

Notice the queue of start times. The model usually calls `get_weather` for both
cities **in the same turn**, so the two calls overlap. Our first version stored
one start time per tool name, and the second call overwrote the first, which
gave nonsense durations. Hooks see concurrent events, so key your
bookkeeping accordingly.

**Hooks vs trace processors:** both see every step. Hooks are for **acting
during the run** (metrics, alerts, updating a progress bar in your app, enforcing
a budget). Processors are for **recording** the finished structure. Keep hooks
fast: they run inside the agent loop.

## 5 · Common mistakes

- **Leaving the default tracing exporter on without a key**, or with sensitive
  data you didn't mean to share. Decide deliberately (see `common.py`).
- **Showing every raw event to users.** Filter: tokens for the answer, short
  status lines for tools and handoffs.
- **Forgetting `trace(...)` around multi-run workflows**, so related runs show up as
  unrelated traces.
- **Slow work in hooks.** It slows every step; queue it instead.
- **Reading `final_output` before the stream is finished.** It's only complete
  after `stream_events()` ends.

## 6 · Check yourself

<details><summary><b>1.</b> Which stream event would you use for a typing effect, and which for "checking the weather…"?</summary>

`raw_response_event` with `ResponseTextDeltaEvent` data (its `.delta`) for the typing effect; `run_item_stream_event` with name `tool_called` for the status line.
</details>

<details><summary><b>2.</b> What's the difference between a trace and a span?</summary>

A trace is one end-to-end workflow (possibly several runs, grouped with `trace()`). Spans are the nested operations inside it: agent turns, model generations, tool calls, handoffs, guardrails.
</details>

<details><summary><b>3.</b> How do you keep traces on your own machine?</summary>

Install your own processor with `set_trace_processors([...])` (which replaces the default OpenAI exporter), and enable tracing. The processor decides where spans go: memory, a file, or your own observability stack.
</details>

<details><summary><b>4.</b> Hooks or a trace processor to stop a run when it exceeds a token budget?</summary>

Hooks: they run during the run (e.g. `on_llm_end` can track tokens and raise when the budget is exceeded). Trace processors record; they aren't meant to steer execution.
</details>

<details><summary><b>5.</b> What does <code>trace_include_sensitive_data=False</code> change?</summary>

Spans keep their structure and timings, but model inputs/outputs and tool arguments/results are left out, which matters when traces go to a hosted service.
</details>

## Takeaway

**Stream** to users (tokens plus status lines), **trace** for developers (a
span tree you can keep entirely local), and **hook** into the loop when your code
must react as the run happens. All three see the same run; they differ in who
consumes the information, and when.

Next → `11_mcp_tools.ipynb`: giving agents tools from MCP servers.